# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

In [2]:
!pip -q install duckdb pandas pyarrow huggingface_hub

In [3]:
import duckdb
import pandas as pd
from huggingface_hub import login

login(token=HF_TOKEN)

## 1. Unit of Analysis + Time Window

**Unit of Analysis (One Row):**

One row represents the daily performance of a single content item for a specific client on a specific report date.

**Time Window:**

For this assignment, I use data from **March 2026 (2026-03)** as the analysis window, following the assignment recommendation to work on a mid-panel month.

Features:
- GSC impressions
- GSC clicks
- GSC average position
- report_date-derived historical/aggregation information where appropriate
- availability indicator where appropriate

Label / Evaluation outcome:
For the clustering lane, there is no supervised target label.
The primary output is an archetype assignment.

Context:
- report_date
- client_hash_id
- content_hash_id

Excluded:
- future-window performance
- final-month outcome information
- any label-derived information
- client identifiers as model features

In [4]:
!pip -q install duckdb pandas pyarrow huggingface_hub

In [5]:
import duckdb
import pandas as pd
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

con = duckdb.connect()

In [6]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

In [7]:
from huggingface_hub import list_repo_files
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

print(files[:20])

['.gitattributes', 'README.md', 'dim_clients.parquet', 'dim_content.parquet', 'fact_content_daily_performance/month=2025-01/data_0.parquet', 'fact_content_daily_performance/month=2025-02/data_0.parquet', 'fact_content_daily_performance/month=2025-03/data_0.parquet', 'fact_content_daily_performance/month=2025-04/data_0.parquet', 'fact_content_daily_performance/month=2025-05/data_0.parquet', 'fact_content_daily_performance/month=2025-06/data_0.parquet', 'fact_content_daily_performance/month=2025-07/data_0.parquet', 'fact_content_daily_performance/month=2025-08/data_0.parquet', 'fact_content_daily_performance/month=2025-09/data_0.parquet', 'fact_content_daily_performance/month=2025-10/data_0.parquet', 'fact_content_daily_performance/month=2025-11/data_0.parquet', 'fact_content_daily_performance/month=2025-12/data_0.parquet', 'fact_content_daily_performance/month=2026-01/data_0.parquet', 'fact_content_daily_performance/month=2026-02/data_0.parquet', 'fact_content_daily_performance/month=20

In [8]:
REL = """
read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS c
FROM {REL}
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


In [9]:
query = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {REL};
"""

con.sql(query).df()

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [10]:
con.sql("""
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5;
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


# Verification Summary

- The grain query returned no duplicate rows, confirming the unit of analysis.
- The count query verified the number of rows and date range for March 2026.
- The availability query showed how many rows contain valid Google Search Console data using `IS TRUE`.

In [11]:
con.sql(f"""
SELECT *
FROM {REL}
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [12]:
query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    client_has_gsc,
    gsc_data_available
FROM {REL}
WHERE gsc_data_available IS TRUE
LIMIT 20;
"""

feature_df = con.sql(query).df()

feature_df.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,client_has_gsc,gsc_data_available
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,67,True,True
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0,True,True
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,616,True,True
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,28,True,True
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,25,True,True



### 1. gsc_impressions
Knowable at the decision moment because it is already recorded before making the prediction.

### 2. gsc_clicks
Knowable at the decision moment because it comes from historical Search Console data.

### 3. gsc_sum_position
Knowable at the decision moment because the search position has already been observed.

### 4. client_has_gsc
Knowable at the decision moment because client access status is already known.

### 5. gsc_data_available
Knowable at the decision moment because it indicates whether Search Console data exists before prediction.

In [15]:
honest = feature_df[
    [
        "gsc_impressions",
        "gsc_sum_position",
        "client_has_gsc",
        "gsc_data_available"
    ]
]

honest.head()

,gsc_impressions,gsc_sum_position,client_has_gsc,gsc_data_available
0,20,67,True,True
1,1,0,True,True
2,125,616,True,True
3,7,28,True,True
4,11,25,True,True


In [16]:
leak = feature_df[
    [
        "gsc_impressions",
        "gsc_sum_position",
        "gsc_clicks"
    ]
]

leak.head()

,gsc_impressions,gsc_sum_position,gsc_clicks
0,20,67,0
1,1,0,0
2,125,616,1
3,7,28,0
4,11,25,0


# Leakage Demonstration

The column **gsc_clicks** is treated as the target in this example.

Including it as an input feature would allow the model to directly see the answer, producing unrealistically high performance.

After removing this column, the model must rely only on historical information, producing a more honest evaluation.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [17]:
query = f"""
SELECT
    gsc_data_available,
    COUNT(*) AS rows
FROM {REL}
GROUP BY gsc_data_available;
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_data_available,rows
0,False,6230317
1,True,3611061


In [18]:
query = """
SELECT
    MIN(gsc_data_start) AS earliest_start,
    MAX(gsc_data_start) AS latest_start
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
);
"""

con.sql(query).df()

,earliest_start,latest_start
0,2025-01-27,2026-06-02


In [19]:
content_check = con.sql(f"""
SELECT
    content_hash_id,
    COUNT(*) AS observed_days,
    AVG(gsc_impressions) AS avg_impressions,
    AVG(gsc_clicks) AS avg_clicks,
    AVG(gsc_avg_position) AS avg_position
FROM {REL}
WHERE gsc_data_available IS TRUE
GROUP BY content_hash_id
HAVING COUNT(*) >= 3
LIMIT 10
""").df()

content_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,observed_days,avg_impressions,avg_clicks,avg_position
0,content_7a105f548d9c6916,31,210.419355,0.225806,7.209549
1,content_a3ea9792f793ec72,31,14.612903,0.000000,2.987198
2,content_36c36abc7650d7af,31,181.612903,0.193548,6.724039
3,content_a7da352b73b02668,31,159.483871,0.419355,7.244844
4,content_1855a661b4d36130,31,13.838710,0.032258,4.209227
5,content_5d412fba6e1a2582,31,7.193548,0.032258,9.445635
6,content_1f380a642aed423b,31,3.096774,0.032258,6.014516
7,content_22c063002b7c1caf,31,10.129032,0.032258,9.155335
8,content_aafb2ab7e5fc80d0,31,248.677419,0.645161,5.258331
9,content_20403327d8d9374c,31,114.870968,0.322581,8.834415


In [20]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

demo = feature_df[
    ["gsc_impressions", "gsc_sum_position", "gsc_clicks"]
].dropna()

y = (demo["gsc_clicks"] > 0).astype(int)

X_honest = demo[["gsc_impressions", "gsc_sum_position"]]
X_leak = demo[["gsc_impressions", "gsc_sum_position", "gsc_clicks"]]

model_honest = DecisionTreeClassifier(max_depth=3, random_state=42)
model_honest.fit(X_honest, y)

model_leak = DecisionTreeClassifier(max_depth=3, random_state=42)
model_leak.fit(X_leak, y)

honest_pred = model_honest.predict(X_honest)
leak_pred = model_leak.predict(X_leak)

print("Honest F1:", f1_score(y, honest_pred))
print("Leaky F1:", f1_score(y, leak_pred))

Honest F1: 1.0
Leaky F1: 1.0


March 2026 is a single-month analysis window, so the archetypes may not represent longer-term content behaviour.

## 4. Data Limits

### Observed Limitation

The data history is not balanced across all clients. Some clients started providing Google Search Console data earlier than others.

Some rows also have `gsc_data_available = FALSE`, meaning Search Console metrics are unavailable for those records. These rows should not be interpreted as having zero search performance.

# Limitation

This notebook analyses only March 2026 and excludes rows where Google Search Console data is unavailable. Therefore, the results may not represent all clients or longer-term behaviour.

# Self Check

- ✅ Unit of analysis defined
- ✅ Time window defined
- ✅ Features, label, context and excluded fields documented
- ✅ Grain verified
- ✅ Row count verified
- ✅ Availability verified using `IS TRUE`
- ✅ Five-feature frame created
- ✅ Leakage example demonstrated
- ✅ One limitation documented